In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
print(os.listdir('/content/drive/MyDrive/Ketastasia/data'))

['Penn_Action.tar.gz', 'kaggle.zip', 'combined_dataset_p1_13.csv']


In [ ]:
!tar -xzf '/content/drive/MyDrive/Ketastasia/data/Penn_Action.tar.gz' \
    -C '/content/'

!unzip -o -q '/content/drive/MyDrive/Ketastasia/data/kaggle.zip' \
    -d '/content/Kaggle'

print("=== Penn Action ===")
!ls /content/Penn_Action/

print("\n=== Kaggle ===")
!ls /content/Kaggle/

=== Penn Action ===
frames	labels	README	tools

=== Kaggle ===
'barbell biceps curl'  'lateral raise'	    'russian twist'
'bench press'	       'lat pulldown'	    'shoulder press'
'chest fly machine'    'leg extension'	     squat
 deadlift	       'leg raises'	    't bar row'
'decline bench press'   plank		    'tricep dips'
'hammer curl'	       'pull Up'	    'tricep Pushdown'
'hip thrust'	        push-up
'incline bench press'  'romanian deadlift'


In [ ]:
!pip install "protobuf>=5.26,<6" mediapipe -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 87.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 9.6 MB/s eta 0:00:00


In [ ]:
import mediapipe as mp
print(mp.__version__)
import numpy as np
import pandas as pd
import scipy.io as sio
import cv2
import os
import glob

0.10.35


In [ ]:
import scipy.io as sio
import glob, os
import pandas as pd

FITNESS_ACTIONS = ['squat','pushup','pullup','jumping_jacks','situp',
                    'bench_press','jump_rope','clean_and_jerk']

LABELS_DIR = '/content/Penn_Action/labels'
penn_video_rows = []

for mat_file in sorted(glob.glob(f'{LABELS_DIR}/*.mat')):
    ann = sio.loadmat(mat_file)
    action = ann['action'][0].strip()
    if action not in FITNESS_ACTIONS:
        continue
    video_id = os.path.basename(mat_file).replace('.mat', '')
    penn_video_rows.append({
        'video_id': video_id,
        'source': 'penn',
        'label': action,
        'mat_file': mat_file
    })

penn_video_meta = pd.DataFrame(penn_video_rows)
print(f"Penn video: {len(penn_video_meta)}")
print(penn_video_meta['label'].value_counts())

Penn video: 1163
label
squat             231
pushup            211
pullup            199
bench_press       140
jumping_jacks     112
situp             100
clean_and_jerk     88
jump_rope          82
Name: count, dtype: int64


In [ ]:
import scipy.io as sio
import glob

all_actions = set()
for mat_file in sorted(glob.glob('/content/Penn_Action/labels/*.mat')):
    ann = sio.loadmat(mat_file)
    action = ann['action'][0].strip()
    all_actions.add(action)

print(f"სულ unique actions: {len(all_actions)}")
for a in sorted(all_actions):
    print(f"  '{a}'")

სულ unique actions: 15
  'baseball_pitch'
  'baseball_swing'
  'bench_press'
  'bowl'
  'clean_and_jerk'
  'golf_swing'
  'jump_rope'
  'jumping_jacks'
  'pullup'
  'pushup'
  'situp'
  'squat'
  'strum_guitar'
  'tennis_forehand'
  'tennis_serve'


In [ ]:
# ჯერ ნახე რა არის Penn_Action ფოლდერში
!ls /content/Penn_Action/

# შემდეგ labels ფოლდერი არსებობს თუ არა
!ls /content/Penn_Action/labels/ | head -20

# რამდენი .mat ფაილია სულ
!find /content/Penn_Action -name "*.mat" | wc -l

frames	labels	README	tools
0001.mat
0002.mat
0003.mat
0004.mat
0005.mat
0006.mat
0007.mat
0008.mat
0009.mat
0010.mat
0011.mat
0012.mat
0013.mat
0014.mat
0015.mat
0016.mat
0017.mat
0018.mat
0019.mat
0020.mat
2326


In [ ]:
import os

for root, dirs, files in os.walk('/content/Kaggle'):
    level = root.replace('/content/Kaggle', '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 2:
        for f in files[:3]:
            print(f'{indent}  {f}')

Kaggle/
  leg raises/
    leg raises_2.MOV
    leg raises_8.mp4
    leg raises_6.MOV
  squat/
    squat_21.mp4
    squat_3.MOV
    squat_4.MOV
  barbell biceps curl/
    barbell biceps curl_17.mp4
    barbell biceps curl_51.mp4
    barbell biceps curl_21.mp4
  chest fly machine/
    chest fly machine_27.mp4
    chest fly machine_16.mp4
    chest fly machine_17.mp4
  leg extension/
    leg extension_17.mp4
    leg extension_24.mp4
    leg extension_23.mp4
  lateral raise/
    lateral raise_16.mp4
    lateral raise_3.MOV
    lateral raise_6.MOV
  bench press/
    bench press_4.mp4
    bench press_42.mp4
    bench press_58.mp4
  hammer curl/
    hammer curl_14.mp4
    hammer curl_10.mp4
    hammer curl_3.MOV
  decline bench press/
    dbp_9.mp4
    dbp_4.MOV
    dbp_1.mp4
  lat pulldown/
    lat pulldown_22.mp4
    lat pulldown_30.mp4
    lat pulldown_47.mp4
  incline bench press/
    incline bench press_24.mp4
    incline bench press_33.mp4
    incline bench press_11.mp4
  tricep Pushdow

In [ ]:
import urllib.request

model_url = "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/1/pose_landmarker_lite.task"
model_path = "/content/pose_landmarker.task"

if not os.path.exists(model_path):
    urllib.request.urlretrieve(model_url, model_path)
    print("მოდელი ჩამოიტვირთა ✅")

from mediapipe.tasks import python
from mediapipe.tasks.python import vision

base_options = python.BaseOptions(model_asset_path=model_path)
options = vision.PoseLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.IMAGE
)
detector = vision.PoseLandmarker.create_from_options(options)
print("detector მზადაა ✅")

მოდელი ჩამოიტვირთა ✅
detector მზადაა ✅


In [ ]:
LABEL_MAP = {
    'squat': 'squat', 'push-up': 'pushup', 'pull Up': 'pullup',
    'bench press': 'bench_press', 'leg raises': 'leg_raises',
    'barbell biceps curl': 'bicep_curl', 'chest fly machine': 'chest_fly',
    'leg extension': 'leg_extension', 'lateral raise': 'lateral_raise',
    'hammer curl': 'hammer_curl', 'decline bench press': 'decline_bench_press',
    'lat pulldown': 'lat_pulldown', 'incline bench press': 'incline_bench_press',
    'tricep Pushdown': 'tricep_pushdown', 'romanian deadlift': 'romanian_deadlift',
    'deadlift': 'deadlift', 't bar row': 't_bar_row', 'russian twist': 'russian_twist',
    'hip thrust': 'hip_thrust', 'plank': 'plank', 'shoulder press': 'shoulder_press',
    'tricep dips': 'tricep_dips'
}

KAGGLE_DIR = '/content/Kaggle'
kaggle_video_rows = []

action_folders = [f for f in os.listdir(KAGGLE_DIR)
                   if os.path.isdir(os.path.join(KAGGLE_DIR, f))]

for action_folder in action_folders:
    action_path = os.path.join(KAGGLE_DIR, action_folder)
    label = LABEL_MAP.get(action_folder, action_folder.lower().replace(' ', '_'))
    video_files = (glob.glob(f'{action_path}/*.mp4') +
                   glob.glob(f'{action_path}/*.MOV'))
    for video_file in video_files:
        video_id = os.path.basename(video_file).rsplit('.', 1)[0]
        kaggle_video_rows.append({
            'video_id': video_id,
            'source': 'kaggle',
            'label': label,
            'video_file': video_file
        })

kaggle_video_meta = pd.DataFrame(kaggle_video_rows)
print(f"Kaggle video: {len(kaggle_video_meta)}")
print(kaggle_video_meta['label'].value_counts())

Kaggle video: 650
label
bicep_curl             62
bench_press            61
pushup                 56
lat_pulldown           51
tricep_pushdown        50
lateral_raise          37
incline_bench_press    33
deadlift               32
squat                  29
chest_fly              28
pullup                 26
leg_extension          25
leg_raises             21
t_bar_row              21
tricep_dips            20
hammer_curl            19
hip_thrust             18
shoulder_press         17
russian_twist          13
decline_bench_press    12
romanian_deadlift      12
plank                   7
Name: count, dtype: int64


In [ ]:
video_meta = pd.concat([
    penn_video_meta[['video_id','source','label']],
    kaggle_video_meta[['video_id','source','label']]
], ignore_index=True)

video_meta_full = video_meta.copy()

print(f"Full:        {len(video_meta_full)} video, {video_meta_full['label'].nunique()} class")

Full:        1813 video, 26 class


In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

def make_video_split(meta_df, random_state=42):
    sgkf_test = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=random_state)
    train_val_idx, test_idx = next(sgkf_test.split(
        meta_df, meta_df['label'], groups=meta_df['video_id']
    ))
    train_val_meta = meta_df.iloc[train_val_idx].reset_index(drop=True)
    test_meta = meta_df.iloc[test_idx].reset_index(drop=True)

    sgkf_val = StratifiedGroupKFold(n_splits=6, shuffle=True, random_state=random_state)
    train_idx, val_idx = next(sgkf_val.split(
        train_val_meta, train_val_meta['label'], groups=train_val_meta['video_id']
    ))
    train_meta = train_val_meta.iloc[train_idx]
    val_meta = train_val_meta.iloc[val_idx]

    split_map = {}
    split_map.update({vid: 'train' for vid in train_meta['video_id']})
    split_map.update({vid: 'val'   for vid in val_meta['video_id']})
    split_map.update({vid: 'test'  for vid in test_meta['video_id']})
    return split_map

split_map = make_video_split(video_meta_full)

check_df = video_meta_full.copy()
check_df['split'] = check_df['video_id'].map(split_map)
print(check_df.groupby('split')['video_id'].nunique())
print(check_df.groupby(['split','label']).size().unstack(fill_value=0))

split
test      259
train    1295
val       259
Name: video_id, dtype: int64
label  bench_press  bicep_curl  chest_fly  clean_and_jerk  deadlift  \
split                                                                 
test            29           9          3              12         4   
train          143          44         21              64        24   
val             29           9          4              12         4   

label  decline_bench_press  hammer_curl  hip_thrust  incline_bench_press  \
split                                                                      
test                     3            3           2                    5   
train                    7           13          13                   24   
val                      2            3           3                    4   

label  jump_rope  ...  pullup  pushup  romanian_deadlift  russian_twist  \
split             ...                                                     
test          13  ...      33      3

In [ ]:
manifest_df = video_meta_full.copy()
manifest_df['split'] = manifest_df['video_id'].map(split_map)
manifest_df.to_csv('/content/drive/MyDrive/Ketastasia/data/combined.csv', index=False)
print(manifest_df.head())

  video_id source        label  split
0     0341   penn  bench_press    val
1     0342   penn  bench_press  train
2     0343   penn  bench_press  train
3     0344   penn  bench_press  train
4     0345   penn  bench_press  train
